# K-Means Clustering — Mall Customer Segmentation

This notebook applies K-Means clustering to segment mall customers by age, annual income,
and spending score, revealing natural purchasing behaviour groups without any labelled data.

- **Elbow method** identifies the optimal number of clusters by plotting inertia vs $k$
- **2-D scatter** of annual income vs spending score visualises cluster structure
- **Pair plot** shows all pairwise feature combinations coloured by cluster assignment
- **Cluster profile table** summarises mean feature values per segment for business interpretation
- Results demonstrate that K-Means recovers interpretable customer archetypes from continuous features

## Mathematical Intuition

### K-Means Objective

K-Means minimises the within-cluster sum of squared distances (inertia):

$$J = \sum_{k=1}^{K} \sum_{i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

where $C_k$ is the set of points assigned to cluster $k$ and $\boldsymbol{\mu}_k$ is its centroid.

### Algorithm (Lloyd's Method)

The algorithm alternates two steps until convergence:

**E-step (Assign):** Assign each point to its nearest centroid:
$$z_i = \underset{k}{\operatorname{argmin}}\; \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

**M-step (Update):** Recompute each centroid as the mean of its assigned points:
$$\boldsymbol{\mu}_k = \frac{1}{|C_k|} \sum_{i \in C_k} \mathbf{x}_i$$

Each iteration is guaranteed not to increase $J$, so the algorithm converges (to a local minimum).

### Elbow Method

Plot $J$ vs $K$. $J$ decreases monotonically with $K$ (adding clusters always reduces inertia).
The "elbow" — the $K$ where the rate of decrease sharply slows — is the natural choice.
Beyond the elbow, adding clusters yields diminishing returns.

## Dataset Overview

**Source:** Mall Customer Segmentation | **URL:** dsrscientist/dataset1 on GitHub | **Rows:** 200

| Feature | Type | Description |
|---|---|---|
| CustomerID | Integer | Unique customer identifier (dropped) |
| Genre | Categorical | Customer gender (dropped) |
| Age | Continuous | Customer age in years |
| Annual Income (k$) | Continuous | Annual income in thousands of USD |
| Spending Score (1-100) | Continuous | Mall-assigned spending score (1 = low, 100 = high) |

**Clustering features:** Age, Annual Income (k$), Spending Score (1-100)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

from mlpackage import KMeans, StandardScaler

url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/mall_customers.csv"
df  = pd.read_csv(url)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

## Exploratory Data Analysis

In [ ]:
print(df.dtypes)
print()
print(df.describe())

In [ ]:
feature_cols = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["steelblue", "coral", "seagreen"]
for ax, col, color in zip(axes, feature_cols, colors):
    ax.hist(df[col], bins=20, color=color, edgecolor="white")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
corr = df[feature_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
X_raw = df[feature_cols].values

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print("Scaled mean (approx 0):", X_scaled.mean(axis=0).round(6))
print("Scaled std  (approx 1):", X_scaled.std(axis=0).round(6))

## Elbow Method

In [ ]:
k_range  = list(range(1, 11))
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker="o", color="steelblue")
plt.axvline(x=5, color="red", linestyle="--", linewidth=1.2, label="Elbow (k=5)")
plt.title("Elbow Method: Inertia vs Number of Clusters")
plt.xlabel("k (number of clusters)")
plt.ylabel("Inertia (within-cluster SSE)")
plt.legend()
plt.tight_layout()
plt.show()

## Model Training — k=5

In [ ]:
km = KMeans(n_clusters=5, random_state=42)
km.fit(X_scaled)

labels   = km.labels_
centers  = km.cluster_centers_

print(f"Inertia  : {km.inertia_:.4f}")
print(f"Iterations: {km.n_iter_}")
print(f"Cluster sizes: { {k: int((labels==k).sum()) for k in range(5)} }")

## Evaluation and Visualizations

In [ ]:
# 2-D scatter: Annual Income vs Spending Score, coloured by cluster
income_idx  = feature_cols.index("Annual Income (k$)")
spend_idx   = feature_cols.index("Spending Score (1-100)")

# Use original (unscaled) values for interpretability
income_vals = X_raw[:, income_idx]
spend_vals  = X_raw[:, spend_idx]
center_income = scaler.inverse_transform(centers)[:, income_idx]
center_spend  = scaler.inverse_transform(centers)[:, spend_idx]

palette = ["steelblue", "coral", "seagreen", "purple", "goldenrod"]

plt.figure(figsize=(9, 6))
for k in range(5):
    mask = labels == k
    plt.scatter(income_vals[mask], spend_vals[mask], color=palette[k], s=40,
                alpha=0.7, label=f"Cluster {k}")
plt.scatter(center_income, center_spend, color="black", s=200, marker="X",
            zorder=6, label="Centroids")
plt.title("K-Means Clusters: Annual Income vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Pair plot (manual 3x3 subplot grid)
fig, axes = plt.subplots(3, 3, figsize=(12, 10))

for i, col_i in enumerate(feature_cols):
    for j, col_j in enumerate(feature_cols):
        ax = axes[i][j]
        if i == j:
            for k in range(5):
                mask = labels == k
                ax.hist(X_raw[mask, i], bins=12, color=palette[k], alpha=0.5)
            ax.set_title(col_i, fontsize=8)
        else:
            for k in range(5):
                mask = labels == k
                ax.scatter(X_raw[mask, j], X_raw[mask, i],
                           color=palette[k], s=15, alpha=0.6, label=f"C{k}")
        if i == 2:
            ax.set_xlabel(col_j, fontsize=7)
        if j == 0:
            ax.set_ylabel(col_i, fontsize=7)

axes[0][2].legend(fontsize=6, bbox_to_anchor=(1.05, 1), loc="upper left")
fig.suptitle("Pair Plot of Clustering Features Coloured by Cluster", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profile table
df_plot           = df[feature_cols].copy()
df_plot["Cluster"] = labels

profile = df_plot.groupby("Cluster")[feature_cols].mean().round(2)
profile.index.name = "Cluster"
print("Cluster Profile (feature means):")
print(profile.to_string())

## Interpretation and Conclusions

- **The elbow plot clearly bends at $k = 5$**, confirming that five clusters capture the dominant structure in the data; beyond this point each additional cluster yields a much smaller reduction in inertia.
- **The Annual Income vs Spending Score scatter reveals five distinct customer archetypes:** high-income/high-spending (prime targets for premium offers), high-income/low-spending (cautious savers), low-income/high-spending (impulsive buyers), low-income/low-spending (budget-conscious), and average-income/average-spending (the largest middle segment).
- **The cluster profile table quantifies these archetypes numerically**, enabling marketing teams to craft targeted campaigns — for example, loyalty rewards for high-income/high-spending customers versus discount coupons for the low-income/high-spending group.
- **K-Means assumes spherical, equal-variance clusters** and is sensitive to feature scaling; without `StandardScaler` the algorithm would be dominated by `Annual Income (k$)` due to its larger numeric range, masking the spending-score structure.
- **Inertia measures compactness but not separation**; complementary metrics such as silhouette score would provide a fuller picture of cluster quality, and DBSCAN (Notebook 12) handles non-spherical clusters that K-Means would incorrectly split.